# Colorado stream/river flow — liberation pipeline

Thin orchestration over the tested `streamflow` package. Runs four stages:
**retrieve → audit → cleanup → publish**. The package holds all the logic; this
notebook is the human-facing narrative. Implements issue [#10](https://github.com/CUPIDS-Lab/co-environmental-data/issues/10);
stamped out from the sibling `reservoir-storage` pipeline (#9).

**⚠️ The two sources overlap:** DWR re-serves many USGS gages (joined by
`usgs_site_no`). They co-exist as distinct rows; de-duplicate to one source for
single-series analysis. The overlap is also a built-in accuracy check
(`reconcile_cross_source`). See `docs/survey-notes.md` and `data/lookups/concepts.yaml`.

In [ ]:
# Config
MODE = "live"          # "live" = fetch the APIs | "demo" = offline 1-gage sample
FRESH = True           # True clears data/original first so the CSV reflects exactly this run
SOURCES = None         # None = both | ["usgs_nwis"] | ["dwr_cdss"]
# Sampling hook: a handful of USGS site numbers (also pulls their DWR mirrors).
# Set SITES = None for the full curated seed (33 gages × 2 sources).
SITES = {"09095500", "09152500", "08220000", "06714000", "07091200"}

from streamflow import fetch, audit, clean, config
import pandas as pd
print("sources:", list(config.get_sources()))

## 1. Retrieve — fetch raw responses → immutable `data/original/` + manifest

`fetch_all` is idempotent (skips cached files), resilient (404/no-data and throttles
are durable, not fatal), and reports progress. **CDSS throttles a large DWR burst
with HTTP 403** — for a clean full DWR pull set `CDSS_API_KEY` and/or
`STREAMFLOW_RATE_LIMIT=3` in the environment.

In [ ]:
import shutil
if FRESH and config.ORIGINAL.exists():
    shutil.rmtree(config.ORIGINAL)
config.ORIGINAL.mkdir(parents=True, exist_ok=True)

if MODE == "live":
    artifacts = fetch.fetch_all(sources=SOURCES, sites=SITES)
    print(f"fetched {len(artifacts)} series")
else:
    # offline demo: seed one USGS fixture
    src = config.get_sources()["usgs_nwis"]
    art = next(a for a in src.discover(sites={"09095500"}))
    art.local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(config.PROJECT_DIR / "tests" / "fixtures" / "usgs_nwis_sample.json", art.local_path)
    print("demo seed written")

## 2. Audit (raw) — did the fetch return data?

In [ ]:
print(audit.profile_raw())

## 3. Cleanup — parse → tidy long → validate → `data/processed/streamflow.csv`

In [ ]:
data, prov = clean.run(sources=SOURCES, sites=SITES, fail_on_empty=True)
print(f"{len(data):,} rows · {data.site_id.nunique()} gages · sources={sorted(data.source.unique())}")
data.head()

## 4. Audit (processed) — profile, per-gage coverage, and cross-source reconciliation

`reconcile_cross_source` is this pipeline's signature accuracy check: USGS and DWR
re-serve the same gages, so they should agree on shared dates (~100 %).

In [ ]:
print(audit.audit_processed())
cov = audit.coverage_report()
audit.variables_report()
xs = audit.reconcile_cross_source()
print("\nCross-source agreement (USGS vs DWR re-serve):")
xs

## 5. Publish — finalize the deliverable

The processed CSV is the deliverable. Optionally spot-check the latest values
against the agency pages (`audit.reconcile`), then deposit to Dataverse for a DOI
(`dataverse/DEPOSIT.md`). The headless twin `python -m streamflow.pipeline` runs
all of the above for CI.

In [ ]:
# Optional manual spot-check vs the agency current-conditions pages (cfs):
expected = {
    # ("usgs_nwis", "09095500"): 12400.0,   # refresh from waterdata.usgs.gov before running
}
if expected:
    display(audit.reconcile(expected))
print("deliverable:", config.CANONICAL_CSV)